In [1]:
!nvidia-smi

Mon Feb 23 21:05:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
REPO_URL = "https://github.com/ceus90/CS224N-TheEfficiencyThreshold.git"
REPO_DIR = "CS224N-TheEfficiencyThreshold"

!git clone {REPO_URL}
%cd {REPO_DIR}

Cloning into 'CS224N-TheEfficiencyThreshold'...
remote: Enumerating objects: 186, done.
remote: Counting objects: 100% (186/186), done.
remote: Compressing objects: 100% (107/107), done.
remote: Total 186 (delta 83), reused 149 (delta 65), pack-reused 0 (from 0)
Receiving objects: 100% (186/186), 1.52 MiB | 19.73 MiB/s, done.
Resolving deltas: 100% (83/83), done.
/content/CS224N-TheEfficiencyThreshold


In [3]:
# switch to abi_splits branch
!git checkout abi_splits
!git pull

Branch 'abi_splits' set up to track remote branch 'abi_splits' from 'origin'.
Switched to a new branch 'abi_splits'
Already up to date.


In [4]:
!ls data/splits_out/superglue_rte

manifest.json	  N16_seed0.jsonl   N256_seed1.jsonl  N32_seed2.jsonl
N128_seed0.jsonl  N16_seed1.jsonl   N256_seed2.jsonl  N64_seed0.jsonl
N128_seed1.jsonl  N16_seed2.jsonl   N32_seed0.jsonl   N64_seed1.jsonl
N128_seed2.jsonl  N256_seed0.jsonl  N32_seed1.jsonl   N64_seed2.jsonl


In [5]:
from pathlib import Path

SPLITS_DIR = Path("data/splits_out/superglue_rte")

assert SPLITS_DIR.exists(), f"Missing splits dir: {SPLITS_DIR}"
print("Found split files:", sorted([p.name for p in SPLITS_DIR.glob("*.jsonl")])[:10])

Found split files: ['N128_seed0.jsonl', 'N128_seed1.jsonl', 'N128_seed2.jsonl', 'N16_seed0.jsonl', 'N16_seed1.jsonl', 'N16_seed2.jsonl', 'N256_seed0.jsonl', 'N256_seed1.jsonl', 'N256_seed2.jsonl', 'N32_seed0.jsonl']


In [6]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

BASE_MODEL = "Qwen/Qwen3-8B"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loaded:", BASE_MODEL)
print("Device:", next(model.parameters()).device)

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Loaded: Qwen/Qwen3-8B
Device: cuda:0


In [8]:
from datasets import load_dataset

LABELS = ["entailment", "not_entailment"]

def format_rte_input(premise: str, hypothesis: str) -> str:
    return f"Premise: {premise}\nHypothesis: {hypothesis}"

ds_val = load_dataset("super_glue", "rte", split="validation")

eval_examples = []
for row in ds_val:
    x = format_rte_input(row["premise"], row["hypothesis"])
    gold = LABELS[int(row["label"])]
    eval_examples.append({"x": x, "gold": gold})

print("Eval size:", len(eval_examples))
print("First:", eval_examples[0])

README.md: 0.00B [00:00, ?B/s]

rte/train-00000-of-00001.parquet:   0%|          | 0.00/586k [00:00<?, ?B/s]

rte/validation-00000-of-00001.parquet:   0%|          | 0.00/69.8k [00:00<?, ?B/s]

rte/test-00000-of-00001.parquet:   0%|          | 0.00/622k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2490 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/277 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3000 [00:00<?, ? examples/s]

Eval size: 277
First: {'x': 'Premise: Dana Reeve, the widow of the actor Christopher Reeve, has died of lung cancer at age 44, according to the Christopher Reeve Foundation.\nHypothesis: Christopher Reeve had an accident.', 'gold': 'not_entailment'}


In [9]:
import json
from pathlib import Path
from datasets import Dataset

def read_jsonl(path: Path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def make_train_text(ex):
    # Train format: model sees input and must produce the label
    return (
        "Task: Recognize Textual Entailment (RTE).\n"
        "Output EXACTLY one label: entailment or not_entailment.\n\n"
        f"Input:\n{ex['x']}\n"
        f"Label: {ex['label']}"
    )

def build_train_dataset(N: int, seed: int = 0) -> Dataset:
    path = SPLITS_DIR / f"N{N}_seed{seed}.jsonl"
    rows = read_jsonl(path)
    texts = [make_train_text(r) for r in rows]
    return Dataset.from_dict({"text": texts})

# quick sanity check
tmp_ds = build_train_dataset(32, seed=0)
print(tmp_ds[0]["text"])

Task: Recognize Textual Entailment (RTE).
Output EXACTLY one label: entailment or not_entailment.

Input:
Premise: Initially the Bundesbank opposed the introduction of the euro but was compelled to accept it in light of the political pressure of the capitalist politicians who supported its introduction.
Hypothesis: The introduction of the euro has been opposed.
Label: entailment


In [10]:
from transformers import DataCollatorForLanguageModeling

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=1024,   # safe default; can raise if you want
        padding="max_length",
    )

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [11]:
from peft import LoraConfig, get_peft_model, TaskType

def make_lora_model(base_model):
    # Typical attention projection module names
    lora_cfg = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    )
    m = get_peft_model(base_model, lora_cfg)
    m.print_trainable_parameters()
    return m

In [26]:
import time
from transformers import BitsAndBytesConfig,TrainingArguments, Trainer

In [24]:

def train_lora_for_N(N: int, seed: int = 0, out_dir: str = None):
    # Build and tokenize training dataset
    train_ds = build_train_dataset(N, seed=seed).map(
        tokenize_batch, batched=True, remove_columns=["text"]
    )

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )

    # Reload fresh base model for this run
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.float16,
        device_map="auto",
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Attach LoRA adapters
    lora_model = make_lora_model(base)
    lora_model.train()

    # Create output directory for adapter
    if out_dir is None:
        out_dir = f"artifacts/lora_rte_N{N}_seed{seed}"
    Path(out_dir).mkdir(parents=True, exist_ok=True)

    # Training configuration
    # Use a fixed optimization budget across N to make the N-sweep interpretable.
    args = TrainingArguments(
        output_dir=out_dir,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,

        # used fixed max_steps instead of epochs
        max_steps=100,
        num_train_epochs=1,     # ignored when max_steps is set

        learning_rate=2e-4,
        logging_steps=10,
        save_strategy="no",
        fp16=True,
        report_to=[],
        seed=seed,
    )

    # Initialize Trainer
    trainer = Trainer(
        model=lora_model,
        args=args,
        train_dataset=train_ds,
        data_collator=data_collator,
    )

    # Train and measure wall-clock time
    t0 = time.perf_counter()
    trainer.train()
    t1 = time.perf_counter()

    # Save only LoRA adapter weights
    lora_model.save_pretrained(out_dir)
    tokenizer.save_pretrained(out_dir)

    return lora_model, (t1 - t0), out_dir

In [13]:
# Build evaluation prompt (no label)
def build_lora_eval_prompt(x_query: str) -> str:
    return (
        "Task: Recognize Textual Entailment (RTE).\n"
        "Output EXACTLY one label: entailment or not_entailment.\n\n"
        f"Input:\n{x_query}\nLabel:"
    )

# Convert model output into valid RTE label
def parse_label(text: str) -> str:
    t = (text or "").strip().lower()
    if "not_entail" in t or t.startswith("not"):
        return "not_entailment"
    if "entail" in t:
        return "entailment"
    return "unknown"

# Generate one prediction with generation-only timing
@torch.no_grad()
def generate_one(model, prompt: str, max_new_tokens: int = 4):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(model.device)
    prompt_tokens = int(inputs["input_ids"].shape[-1])

    gpu_sync()
    t0 = time.perf_counter()
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    gpu_sync()
    t1 = time.perf_counter()

    gen_ids = out[0][prompt_tokens:]
    decoded = tokenizer.decode(gen_ids, skip_special_tokens=True)
    new_tokens = int(gen_ids.shape[-1])

    return decoded, prompt_tokens, new_tokens, (t1 - t0)

# Ensure accurate GPU timing
def gpu_sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

# Return peak GPU memory usage
def peak_vram_mb():
    return torch.cuda.max_memory_allocated() / (1024**2)

In [14]:
# Evaluate a model on RTE validation and return ICL-compatible metrics dict
def eval_model(model, method: str, model_name: str, N: int, seed: int = 0, max_eval: int = 277):
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    correct = 0
    unknown = 0
    total_gen_time = 0.0
    total_new_tokens = 0
    total_prompt_tokens = 0
    max_prompt_tokens = 0

    subset = eval_examples[:max_eval]
    model.eval()

    for ex in subset:
        prompt = build_lora_eval_prompt(ex["x"])
        decoded, prompt_toks, new_toks, gen_dt = generate_one(model, prompt, max_new_tokens=4)

        pred = parse_label(decoded)

        if pred == "unknown":
            unknown += 1
        if pred == ex["gold"]:
            correct += 1

        total_prompt_tokens += prompt_toks
        max_prompt_tokens = max(max_prompt_tokens, prompt_toks)

        total_new_tokens += new_toks
        total_gen_time += gen_dt

    eval_size = len(subset)
    avg_prompt_tokens = total_prompt_tokens / eval_size
    avg_new_tokens = total_new_tokens / eval_size
    avg_gen_time_sec = total_gen_time / eval_size

    return {
        "method": method,
        "model": model_name,
        "N": N,
        "seed": seed,
        "eval_size": eval_size,

        "accuracy": correct / eval_size,
        "unknown_frac": unknown / eval_size,

        # Speed / latency
        "gen_tokens_per_sec": (total_new_tokens / total_gen_time) if total_gen_time > 0 else 0.0,
        "total_gen_time_sec": total_gen_time,
        "avg_gen_time_sec": avg_gen_time_sec,
        "avg_new_tokens": avg_new_tokens,

        # Prompt length
        "avg_prompt_tokens": avg_prompt_tokens,
        "max_prompt_tokens": max_prompt_tokens,

        # Memory
        "peak_vram_mb": peak_vram_mb(),
    }

In [21]:
!pip install -U bitsandbytes>=0.46.1

In [25]:
# Run selected N values
lora_results = []
for N in [16, 32, 64, 128, 256]:
    lora_model, train_time_sec, out_dir = train_lora_for_N(N, seed=0)

    r = eval_model(
        lora_model,
        method="lora",
        model_name=BASE_MODEL,
        N=N,
        seed=0,
        max_eval=277
    )

    # Add LoRA-only fields
    r["train_time_sec"] = train_time_sec
    r["adapter_dir"] = out_dir

    lora_results.append(r)
    print(r)

Map:   0%|          | 0/16 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.


trainable params: 7,667,712 || all params: 8,198,403,072 || trainable%: 0.0935


OutOfMemoryError: CUDA out of memory. Tried to allocate 32.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 11.81 MiB is free. Including non-PyTorch memory, this process has 14.55 GiB memory in use. Of the allocated memory 14.39 GiB is allocated by PyTorch, and 34.84 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)